# Support Vector Machines (SVM)
In this lecture, we will explore Support Vector Machines (SVM) and their kernelized versions. The SVM is a supervised learning algorithm used for classification and regression tasks.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Hard vs soft margin formulation:__ Formulate the hard- and soft-margin SVM optimization problems using augmented inputs, and interpret the role of slack variables and the penalty parameter $C$ in controlling margin violations.
> * __Hinge-loss objective:__ Convert the constrained soft-margin problem into the unconstrained hinge-loss objective and explain how this changes the optimization approach.
> * __Kernelized dual and support vectors:__ Write the kernelized dual for the augmented-input formulation and use the support-vector decision function to classify new points.

Let's get started!
___

## Examples
Today, we will use the following example to illustrate key concepts:

> [▶ Banknote authentication using SVM classification](CHEME-152-M2-SupportVectorMachine-BanknoteClassification-Ungraded-Codio-Activity.ipynb). In this example, we use a soft-margin SVM with an RBF kernel (via `LIBSVM.jl`) to distinguish genuine banknotes from forgeries on the UCI banknote authentication dataset.
___

## Theory: Support Vector Machine (SVM)
Suppose, we have dataset $\mathcal{D} = \{(\hat{\mathbf{x}}_{i}, y_{i}) \mid i = 1,2,\dots,n\}$, where $\hat{\mathbf{x}}_i \in \mathbb{R}^p$ is an _augmented_ feature vector ($m$ features with additional `1` to model the bias on the end of the vector) and $y_i \in \{-1, 1\}$ is the corresponding class label.

> __What is the goal of an SVM?__ 
> 
> The goal of an SVM is to find the hyperplane $\mathcal{H}(\hat{\mathbf{x}}) = \{\hat{\mathbf{x}} \mid \left<\hat{\mathbf{x}},\theta\right> = 0\}$ that separates the data points into two classes (those points above the hyperplane, and those points below the hyperplane), where $\theta \in \mathbb{R}^{p}$ ($p=m+1$) is the normal vector to the hyperplane, or alternatively, the parameters of the model that we need to estimate.


Support vector machines (SVMs) and other approaches, e.g., [the perceptron](https://en.wikipedia.org/wiki/Perceptron) differ primarily in their optimization objectives and training methods: while a [perceptron](https://en.wikipedia.org/wiki/Perceptron) can find _a hyperplane_ that separates classes, SVMs seek to find the _best hyperplane_ in the sense that the _margin_ between classes is maximized.

There are (at least) two strategies that we could use to estimate the unknown parameters $\theta \in \mathbb{R}^{p}$, depending upon if we know beforehand whether the dataset $\mathcal{D}$ is linearly separable.

Let's start with the case where we know that the dataset is linearly separable.


<div>
    <center>
        <img src="figs/Fig-SVM-Schematic.svg" width="480"/>
    </center>
</div>

### Hard margin case: Linearly separable data
If the data is linearly separable, a hyperplane $\mathcal{H}(\hat{\mathbf{x}})$ exists that perfectly separates the data. We can estimate the _best_ hyperplane by maximizing the _margin_, which is equivalent to minimizing the parameter vector $\theta$.

> __Hard Margin Problem__
>
> The __maximum hard margin problem__ for a support vector classifier is given by:
> $$
\boxed{
\begin{align*}
    \min_{\theta}\quad & \frac{1}{2}\lVert{\theta}\rVert_{2}^{2}\\
    \text{subject to}\quad & y_{i}\left<\hat{\mathbf{x}}_{i},\theta\right> \geq 1\quad\forall i
\end{align*}}
> $$
> where $\theta\in\mathbb{R}^{p}$ denote the unknown parameters that we are trying to estimate, $\hat{\mathbf{x}}_{i}\in\mathbb{R}^{p}$ are the augmented (training) feature vectors, $y_{i}\in\{-1,1\}$ are the class labels, and $p=m+1$ is the number of parameters, where $m$ is the number of features. The index $i$ runs over the training examples, i.e., one constraint per training example.

The hard margin problem is a [quadratic program](https://en.wikipedia.org/wiki/Quadratic_programming) (QP) and can be solved with any off-the-shelf QP solver (e.g., [CVXOPT](https://cvxopt.org/), [OSQP](https://osqp.org/), [Gurobi](https://www.gurobi.com/)). In practice, however, we rarely know beforehand whether the data is linearly separable, so we typically use the more general soft margin formulation discussed next.
___

<div>
    <center>
        <img src="figs/Fig-SVM-Schematic-Softmargin.svg" width="480"/>
    </center>
</div>

### Soft margin case: Not linearly separable
If the data is _not linearly separable_, then we know that a perfect $\mathcal{H}(\hat{\mathbf{x}})$ will not exist, i.e., no hyperplane will separate the data without making at least one mistake. In this case, we can estimate the _best_ hyperplane possible by solving the maximum soft margin problem given by:

> __Soft Margin Problem__
>
> The _maximum soft margin problem_ for a support vector classifier is given by:
> $$
\boxed{
\begin{align*}
    \min_{\theta}\quad & \frac{1}{2}\lVert{\theta}\rVert_{2}^{2} + C\sum_{i=1}^{n}\xi_{i}\\
    \text{subject to}\quad & y_{i}\left<\hat{\mathbf{x}}_{i},\theta\right> \geq 1 - \xi_{i}\quad\forall i\\
    & \xi_{i} \geq 0\quad\forall i
\end{align*}}
> $$
> where $\xi_{i}$ is a _slack variable_ that quantifies the cost of a classification mistake, and $C>0$ is a user-adjustable parameter that controls the trade-off between maximizing the margin and minimizing the slack variables. When $C\gg 1$, mistakes are expensive and the classifier behaves like the hard-margin classifier; when $C\ll 1$, the classifier tolerates more margin violations in exchange for a larger margin.

In practice, we usually solve the equivalent unconstrained [hinge-loss](https://en.wikipedia.org/wiki/Hinge_loss) form:
$$
\min_{\theta}\left[\frac{1}{2}\lVert{\theta}\rVert_{2}^{2} + C\sum_{i=1}^{n}\max\{0, 1 - y_{i}\left<\hat{\mathbf{x}}_{i},\theta\right>\}\right]
$$
where the sum is computed over $n$ training examples. The penalty term is the _hinge loss function_, which penalizes misclassifications and points that fall inside the margin. This unconstrained problem can be solved with [gradient descent](https://en.wikipedia.org/wiki/Gradient_descent), [stochastic gradient descent](https://en.wikipedia.org/wiki/Stochastic_gradient_descent), or off-the-shelf optimization packages.
___

## Kernelized SVM
In the previous sections, we assumed that the data could be separated by a linear hyperplane. However, in many real-world scenarios, the data may not be linearly separable in its original feature space. To address this, we can use the _kernel trick_ to implicitly map the data into a higher-dimensional space where it may be linearly separable.

To apply the kernel trick to the soft margin classifier while keeping the augmented representation, we use the augmented feature vector $\hat{\mathbf{x}}_{i} = [\mathbf{x}_{i};1]$ and let $\phi(\hat{\mathbf{x}})$ denote the feature map associated with the kernel $K(\hat{\mathbf{x}}_{i},\hat{\mathbf{x}}_{j}) = \left<\phi(\hat{\mathbf{x}}_{i}),\phi(\hat{\mathbf{x}}_{j})\right>$. The soft margin problem in feature space becomes:

> __Kernelized Soft Margin Problem (primal)__
>
> The _kernelized soft margin problem_ can be written as:
> $$
\boxed{
\begin{align*}
    \min_{\theta, \xi}\quad & \frac{1}{2}\lVert{\theta}\rVert_{2}^{2} + C\sum_{i=1}^{n}\xi_{i}\\
    \text{subject to}\quad & y_{i}\left<\phi(\hat{\mathbf{x}}_{i}),\theta\right> \geq 1 - \xi_{i}\quad\forall i\\
    & \xi_{i} \geq 0\quad\forall i
\end{align*}}
> $$
> where $\theta$ is the normal vector in feature space (including the bias via the augmented inputs) and $\xi_i$ are the slack variables. Because we use augmented inputs, the bias term is regularized along with the weights. If $b$ is kept separate, the dual includes the equality constraint $\sum_{i=1}^{n}\alpha_i y_i = 0$ and the decision function adds a separate bias term.

In practice, we rarely have an explicit feature map $\phi(\cdot)$ (or it is too high-dimensional to construct), so solving the primal would defeat the purpose of the kernel trick. Instead, we switch to the dual, where the inner products become kernel evaluations:

> __Kernelized Soft Margin Problem (dual)__
>
> The dual (and the one we actually solve) is:
> $$
\boxed{
\begin{align*}
    \max_{\alpha}\quad & \sum_{i=1}^{n}\alpha_{i} - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n}\alpha_{i}\alpha_{j}y_{i}y_{j}K(\hat{\mathbf{x}}_{i},\hat{\mathbf{x}}_{j})\\
    \text{subject to}\quad & 0 \leq \alpha_{i} \leq C\quad\forall i
\end{align*}}
> $$
> where $\alpha_i$ are the Lagrange multipliers (one per training example). The dual is a quadratic program that depends only on the kernel matrix $K(\hat{\mathbf{x}}_{i},\hat{\mathbf{x}}_{j})$, so we can use the same QP machinery as before without ever computing $\phi(\cdot)$ explicitly.

After solving for $\alpha$, we classify new points using:
$$
f(\hat{\mathbf{x}}) = \sum_{i=1}^{n}\alpha_{i}y_{i}K(\hat{\mathbf{x}}_{i},\hat{\mathbf{x}}),\qquad \hat{y} = \text{sign}\{f(\hat{\mathbf{x}})\}
$$
Only points with $\alpha_{i} > 0$ appear in the sum; these are the _support vectors_.

> __KKT reminder.__ Complementary slackness gives $\alpha_i\left[y_i f(\hat{\mathbf{x}}_i) - 1 + \xi_i\right]=0$ and $0\le \alpha_i \le C$. Therefore, if a point satisfies $y_i f(\hat{\mathbf{x}}_i) > 1$ (strictly outside the margin) then $\alpha_i=0$, while points on or inside the margin can have $\alpha_i>0$.
___

## Summary
This lecture formulates SVMs as maximum-margin optimization problems, extends to soft-margin classification with slack variables, and shows how kernelization yields a dual that depends only on kernel evaluations.

> __Key Takeaways:__
>
> * **Hard and soft margin SVMs:** Hard-margin SVMs require every training point to lie on the correct side of the margin and work only when the data is linearly separable. Soft-margin SVMs add slack variables and a penalty parameter to allow controlled margin violations.
> * **Hinge-loss equivalence:** The soft-margin constraints are equivalent to minimizing a hinge-loss objective, enabling unconstrained optimization methods.
> * **Kernelized dual with augmented inputs:** The classifier depends on kernel evaluations and support vectors, and using augmented inputs regularizes the bias term.

These ideas set up kernelized classification models used in later lectures.
___